# Burgers neural surrogates on NVIDIA CUDA

Runs the existing MLP and FNO configurations on an NVIDIA GPU and packages the results. Colab is the supported target.

What runs where:

* **NumPy on the CPU, always:** the pseudo-spectral solver, POD, the POD-Galerkin ROM and the DEIM-ROM. Nothing here moves them to the GPU.
* **PyTorch on CUDA:** MLP and FNO training, evaluation and inference timing, in float32 (no AMP, TF32 off).

Every output goes to `runs/<RUN_ID>/`. The committed CPU/MPS results in `report/` are never written, and the notebook checks the source tree before packaging. A CUDA run is a new backend experiment, not a bit-for-bit reproduction of the MPS checkpoints. See `REMOTE_GPU.md`.

Runtime: **Runtime > Change runtime type > GPU**.

## 0. Configuration

**Data source.** `"upload"` (default) uses the original datasets behind the published MPS results and is the only like-for-like mode. Before running, upload `historical_inputs.zip` to `/content/` (Files panel), or upload it when prompted; either way it is kept outside the repository checkout. Build it on the original machine, from the repository root:

```
zip historical_inputs.zip data/train.npz data/val.npz data/test.npz data/ood_family_test.npz experiments/deim_basis.npz
```

`"regenerate"` rebuilds the data with the current NumPy solver. That is a convenience mode and **not** a reproduction of the historical data: a later snapshot-count fix in `src/solver.py` changes the trajectories by up to about 0.45% of max |u|, so its results must not be compared with the published MPS numbers.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/neural-surrogate-burgers.git"
REF = "main"              # set to the exact commit SHA to test; it must contain src/device.py
RUN_ID_OVERRIDE = None    # optional fixed run name; None -> cuda_<UTC timestamp>. RUN_ID itself is created
                          # once per session (section 3) and is NOT reset by re-running this cell.
DATA_SOURCE = "upload"    # "upload" (like-for-like, default) or "regenerate" (convenience, NOT like-for-like)
HISTORICAL_ZIP = "/content/historical_inputs.zip"  # outside the repository checkout
INPUT_DIR = "/content/inputs"                       # where the upload prompt saves files (outside the checkout)
SEEDS = [0, 1, 2]         # MLP and FNO seeds
RUN_CPU_REFERENCE = True  # also train seed 0 of both models on this machine's CPU, on the same data
RUN_FULL = False          # gate: set True only after both smoke tests (section 6) pass
if not {0, 1, 2} <= set(SEEDS):
    raise ValueError("SEEDS must include 0, 1 and 2: scripts/evaluate_fno_multiseed.py evaluates exactly those")

## 1. Helpers: every command must succeed

In [ ]:
import datetime, hashlib, json, math, os, shutil, subprocess, sys

def run(cmd, log=None):
    """Run a shell command, print its output, and raise on a non-zero exit, so a failure never
    passes silently into the next cell."""
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = p.stdout
    if log:
        with open(log, "w") as f:
            f.write(out)
        out = "\n".join(out.splitlines()[-15:])
    print(out)
    if p.returncode != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")
    return p.stdout

def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def init_run_id(existing, override=None):
    """One identity per notebook session: keep an existing RUN_ID, otherwise create it once."""
    if existing:
        return existing
    return override or "cuda_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def _inside(path, directory):
    path, directory = os.path.realpath(path), os.path.realpath(directory)
    return os.path.commonpath([path, directory]) == directory

def stage_historical_zip(zip_path, input_dir, repo_dir, uploader=None):
    """Return the historical-input archive's path, always outside the repository checkout, or None.
    Colab's upload prompt saves into the working directory, so it is run from input_dir; a copy
    dragged into the checkout by mistake is moved out."""
    if _inside(zip_path, repo_dir) or _inside(input_dir, repo_dir):
        raise ValueError("the historical-input archive must live outside the repository checkout")
    os.makedirs(input_dir, exist_ok=True)
    stray = os.path.join(repo_dir, os.path.basename(zip_path))
    if not os.path.exists(zip_path) and os.path.exists(stray):
        shutil.move(stray, zip_path)
    if not os.path.exists(zip_path) and uploader is not None:
        cwd = os.getcwd()
        os.chdir(input_dir)
        try:
            names = list(uploader())
        finally:
            os.chdir(cwd)
        if names:
            shutil.move(os.path.join(input_dir, names[0]), zip_path)
    return zip_path if os.path.exists(zip_path) else None

GATE_PASSED = False

## 2. GPU and software stack

In [ ]:
run("nvidia-smi")
import platform, torch
print("python ", sys.version.split()[0], "|", platform.platform())
print("torch  ", torch.__version__, "| CUDA (build)", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device: switch the runtime to GPU")
p = torch.cuda.get_device_properties(0)
print("GPU    ", p.name, f"| {p.total_memory / 2**30:.1f} GB | compute capability {p.major}.{p.minor}")

## 3. Clone and check out the explicit ref

In [ ]:
if not os.path.isdir("/content/neural-surrogate-burgers"):
    run(f"git clone -q {REPO_URL} /content/neural-surrogate-burgers")
os.chdir("/content/neural-surrogate-burgers")
run(f"git fetch -q origin && git checkout -q {REF}")
COMMIT = run("git rev-parse HEAD").strip()
DIRTY_AT_START = run("git status --porcelain").strip()
print("commit:", COMMIT, "| dirty at start:", bool(DIRTY_AT_START))
if not os.path.exists("src/device.py") or "--seed" not in open("scripts/train_surrogate.py").read():
    raise RuntimeError(f"{REF} predates CUDA support; set REF to a commit that contains src/device.py")
RUN_ID = init_run_id(globals().get("RUN_ID"), RUN_ID_OVERRIDE)  # stable across re-runs of any cell
RUN = f"runs/{RUN_ID}"
SMOKE, CK, RES, CPU_REF = f"{RUN}/smoke", f"{RUN}/cuda", f"{RUN}/cuda/results", f"{RUN}/cpu_reference"
os.makedirs(RUN, exist_ok=True)
print("run id:", RUN_ID, "| outputs ->", RUN)

A private repository needs a token: store it as a Colab secret and use `https://<token>@github.com/...` as `REPO_URL`.

## 4. Dependencies (the preinstalled CUDA build of PyTorch is kept), then the test suite

In [ ]:
# requirements.txt lists torch unpinned; installing only the other packages avoids pip replacing the
# CUDA wheel with a CPU one.
run("pip install -q numpy scipy matplotlib pytest")
# check in a fresh process, since packages may have changed underneath this kernel
run('python -c "import torch; assert torch.cuda.is_available(), \'CUDA lost after pip install\'; '
    'print(\'CUDA still available:\', torch.version.cuda, torch.cuda.get_device_name(0))"')
run("python -m pytest -q", log=f"{RUN}/pytest.log")

## 5. Data (existing NumPy solver and POD/DEIM, CPU)

`build_deim.py` runs in both modes, so the POD/DEIM offline times come from this machine. It rewrites the committed `report/research/deim_rank_sweep.json`; the new sweep is moved into the run directory and the committed file restored. In upload mode the uploaded historical DEIM basis is then put back, so the ROM/DEIM baseline is the historical one.

In [ ]:
import numpy as np

HISTORICAL_SHA256 = {  # exact files behind the published MPS results
    "data/train.npz":           "233f9eeae594d82f0e97bb53a90d039dbcc4f10c85012fe5a34a086fd88450ee",
    "data/val.npz":             "0d083b211f660215976a52661df28702cf04d991ead8502ca394265f2068346f",
    "data/test.npz":            "a3542f1169912c0749571476640ee7e543dc0d35d61a0c3bb48f829854520e59",
    "data/ood_family_test.npz": "f69a55bd219ae908b89676c45062f967a5975f19f754906f0fe0a0025d206116",
    "experiments/deim_basis.npz": "97f28e8dc6e24e8ec0a34a1f25ecc19e660f3eee36f0333aa1cea2101592cb23",
}
os.makedirs("data", exist_ok=True); os.makedirs("experiments", exist_ok=True)

if DATA_SOURCE == "upload":
    try:
        from google.colab import files
        def uploader():
            print(f"{HISTORICAL_ZIP} not found: choose historical_inputs.zip to upload")
            return files.upload()
    except ImportError:
        uploader = None
    # the archive stays outside the checkout; only its contents (git-ignored data/*.npz and
    # experiments/deim_basis.npz) are extracted into the repository
    if not stage_historical_zip(HISTORICAL_ZIP, INPUT_DIR, os.getcwd(), uploader):
        raise RuntimeError("DATA_SOURCE='upload' needs historical_inputs.zip containing " + ", ".join(HISTORICAL_SHA256)
                           + " (see section 0). Upload it, or set DATA_SOURCE='regenerate' for a NOT like-for-like run.")
    run(f"unzip -o -q {HISTORICAL_ZIP} -d .")
    bad = {p: (sha256(p) if os.path.exists(p) else "missing") for p, h in HISTORICAL_SHA256.items()
           if not os.path.exists(p) or sha256(p) != h}
    if bad:
        raise RuntimeError(f"uploaded files are missing or are not the historical ones: {bad}")
    shutil.copy("experiments/deim_basis.npz", "/tmp/historical_deim_basis.npz")
elif DATA_SOURCE == "regenerate":
    print("WARNING: regenerated data are NOT the historical data; do not compare these results with the published MPS numbers.")
    run("python scripts/generate_dataset.py")
    run("python scripts/generate_ood_dataset.py")
else:
    raise ValueError(DATA_SOURCE)

run("python scripts/build_deim.py", log=f"{RUN}/build_deim.log")
shutil.move("report/research/deim_rank_sweep.json", f"{RUN}/deim_rank_sweep_this_machine.json")
run("git checkout -- report/research/deim_rank_sweep.json")
SWEEP = f"{RUN}/deim_rank_sweep_this_machine.json"

if DATA_SOURCE == "upload":
    rebuilt, hist = np.load("experiments/deim_basis.npz"), np.load("/tmp/historical_deim_basis.npz")
    rebuild_check = {"points_identical": bool(np.array_equal(rebuilt["points"], hist["points"])),
                     "Phi_max_abs_diff_up_to_column_sign": float(np.max(np.abs(np.abs(rebuilt["Phi"]) - np.abs(hist["Phi"]))))}
    print("DEIM basis rebuilt here vs historical:", rebuild_check)
    shutil.copy("/tmp/historical_deim_basis.npz", "experiments/deim_basis.npz")
    assert sha256("experiments/deim_basis.npz") == HISTORICAL_SHA256["experiments/deim_basis.npz"]
else:
    rebuild_check = None

DATA_PROV = {
    "source": DATA_SOURCE,
    "like_for_like_with_historical_mps": DATA_SOURCE == "upload",
    "deim_basis": "uploaded historical" if DATA_SOURCE == "upload" else "rebuilt from regenerated data",
    "deim_rebuild_check": rebuild_check,
    "pod_deim_offline_times": f"{SWEEP} (measured on this machine, NumPy CPU)",
    "files": {p: {"sha256": sha256(p), "is_historical": sha256(p) == h} for p, h in HISTORICAL_SHA256.items()},
}
json.dump(DATA_PROV, open(f"{RUN}/data_provenance.json", "w"), indent=2)
print(json.dumps(DATA_PROV, indent=2))

## 6. CUDA smoke tests, one per model (5 epochs; checks only, not results)

In [ ]:
from scripts.train_surrogate import normalize_params
from src.fno_net import FNO1d
from src.surrogate_net import BurgersSurrogateMLP

TEST = np.load("data/test.npz")

def rel_l2(pred, gt):
    return float(np.linalg.norm(pred - gt) / (np.linalg.norm(gt) + 1e-12))

def verify_model(kind, ck_dir, seed=0, expect="cuda"):
    """Programmatic checks for one trained model; raises on any failure."""
    sfx = "" if seed == 0 else f"_seed{seed}"
    stem = {"mlp": "surrogate", "fno": "fno"}[kind]
    ckpt_path, meta_path = f"{ck_dir}/{stem}_checkpoint{sfx}.pt", f"{ck_dir}/{stem}_run_meta{sfx}.json"
    for path in (ckpt_path, meta_path):
        if not os.path.exists(path):
            raise RuntimeError(f"{kind}: missing {path}")
    meta = json.load(open(meta_path))
    info = meta["device_info"]
    checks = {
        "requested device": meta.get("requested_device") == expect,
        "resolved device": meta["device"] == expect,
        "device info": info["device"] == expect and (expect != "cuda" or (bool(info.get("gpu_name"))
                       and info.get("tf32_matmul") is False and info.get("tf32_cudnn") is False)),
        "finite val loss": math.isfinite(meta["best_val_loss"]),
        "finite train time": math.isfinite(meta["total_train_time_s"]),
        "seed": meta["seed"] == seed,
    }
    raw = torch.load(ckpt_path, weights_only=True)  # no map_location: tensors keep their saved device
    checks["checkpoint saved from device"] = all(t.device.type == expect for t in raw["model_state"].values())
    dev = torch.device(expect)
    if kind == "mlp":
        m = BurgersSurrogateMLP(raw["n_save"], raw["Nx"]).to(dev)
        x = torch.from_numpy(normalize_params(TEST["A"], TEST["nu"])).to(dev)
    else:
        m = FNO1d(raw["Nx"], raw["n_save"], modes=raw["modes"], width=raw["width"], n_layers=raw["n_layers"]).to(dev)
        x = torch.from_numpy(TEST["u"][:, 0].astype(np.float32)).to(dev)
    m.load_state_dict(raw["model_state"]); m.eval()
    with torch.no_grad():
        pred = m(x).cpu().numpy()
    errs = [rel_l2(pred[i], TEST["u"][i]) for i in range(len(pred))]
    checks["finite predictions"] = bool(np.isfinite(pred).all())
    checks["finite test errors"] = all(math.isfinite(e) for e in errs)
    failed = [k for k, ok in checks.items() if not ok]
    if failed:
        raise RuntimeError(f"{kind} seed {seed} ({ck_dir}) failed: {failed}")
    print(f"{kind} seed {seed} on {expect}: all {len(checks)} checks passed | val_loss {meta['best_val_loss']:.3e} | "
          f"test rel L2 {np.mean(errs):.5f} | {meta['total_train_time_s']:.1f} s | {info.get('gpu_name', info['device'])}")
    return {"meta": meta, "test_rel_l2_mean": float(np.mean(errs)), "test_rel_l2_std": float(np.std(errs))}

SMOKE_OK = {"mlp": False, "fno": False, "evaluation": False}

In [ ]:
# MLP smoke test
run(f"python scripts/train_surrogate.py --device cuda --epochs 5 --out-dir {SMOKE}")
verify_model("mlp", SMOKE)
SMOKE_OK["mlp"] = True

In [ ]:
# FNO smoke test
run(f"python scripts/train_fno.py --device cuda --epochs 5 --out-dir {SMOKE}")
verify_model("fno", SMOKE)
SMOKE_OK["fno"] = True

In [ ]:
# evaluation pipeline smoke test (both models, plus the NumPy baselines on the CPU)
def verify_results(res_dir, expect="cuda"):
    r = json.load(open(f"{res_dir}/results.json"))
    vals = [r["in_distribution"][m]["mean"] for m in ("rom", "deim_rom", "mlp", "fno")]
    vals += [r["out_of_family_ic"][m]["mean"] for m in ("rom", "deim_rom", "fno")]
    vals += [r["online_cost_ms"][m] for m in ("mlp", "fno")]
    if not all(math.isfinite(v) for v in vals) or r["device"] != expect:
        raise RuntimeError(f"{res_dir}: non-finite metric or wrong device {r['device']}")
    if r["offline_cost_s"]["mlp_training_200_epochs"] == 8.23:
        raise RuntimeError("the historical MPS MLP training time appeared in a new run")
    print(f"{res_dir}: metrics finite, device {r['device']}; MLP time: {r['offline_cost_provenance']['mlp_training']}")
    return r

run(f"python scripts/evaluate_research_extension.py --device cuda --checkpoint-dir {SMOKE} "
    f"--out-dir {SMOKE}/results --deim-sweep {SWEEP}", log=f"{SMOKE}/evaluate.log")
verify_results(f"{SMOKE}/results")
SMOKE_OK["evaluation"] = True

## 7. Gate

In [ ]:
print("smoke tests:", SMOKE_OK)
if not all(SMOKE_OK.values()):
    raise RuntimeError(f"smoke tests incomplete: {SMOKE_OK}")
if not RUN_FULL:
    raise RuntimeError("Gate (not a failure): both smoke tests passed. Set RUN_FULL = True in section 0, run that cell, then continue from here.")
GATE_PASSED = True

## 8. Full configuration on CUDA: MLP and FNO seeds 0-2 (200 epochs, batch 16, Adam 1e-3, float32)

In [ ]:
assert GATE_PASSED, "run the gate (section 7) first"
FULL = {"mlp": {}, "fno": {}}
for s in SEEDS:
    run(f"python scripts/train_surrogate.py --device cuda --seed {s} --out-dir {CK}", log=f"{CK}/train_mlp_seed{s}.log")
    FULL["mlp"][s] = verify_model("mlp", CK, seed=s)
    run(f"python scripts/train_fno.py --device cuda --seed {s} --out-dir {CK}", log=f"{CK}/train_fno_seed{s}.log")
    FULL["fno"][s] = verify_model("fno", CK, seed=s)

Same-data CPU reference: seed 0 of both models on this machine's CPU, on exactly the data used above.

In [ ]:
assert GATE_PASSED, "run the gate (section 7) first"
CPU = {}
if RUN_CPU_REFERENCE:
    run(f"python scripts/train_surrogate.py --device cpu --out-dir {CPU_REF}", log=f"{RUN}/train_mlp_cpu.log")
    CPU["mlp"] = verify_model("mlp", CPU_REF, expect="cpu")
    run(f"python scripts/train_fno.py --device cpu --out-dir {CPU_REF}", log=f"{RUN}/train_fno_cpu.log")
    CPU["fno"] = verify_model("fno", CPU_REF, expect="cpu")

## 9. Evaluation and CUDA-labelled timing

In [ ]:
assert GATE_PASSED, "run the gate (section 7) first"
run(f"python scripts/evaluate_research_extension.py --device cuda --checkpoint-dir {CK} --out-dir {RES} --deim-sweep {SWEEP}",
    log=f"{RES}/evaluate_research_extension.log")
run(f"python scripts/evaluate_fno_multiseed.py --device cuda --checkpoint-dir {CK} --out {RES}/fno_multiseed_results.json",
    log=f"{RES}/evaluate_fno_multiseed.log")
run(f"python scripts/evaluate_comparison.py --device cuda --checkpoint-dir {CK} --out-dir {RES}/mvp",
    log=f"{RES}/mvp/evaluate_comparison.log")
run(f"python scripts/verify_timing.py --device cuda --checkpoint-dir {CK} --out {RES}/timing_verification_cuda.json",
    log=f"{RES}/verify_timing_cuda.log")
# the same CUDA-trained seed-0 checkpoints timed on this machine's CPU, for a same-session comparison
run(f"python scripts/verify_timing.py --device cpu --checkpoint-dir {CK} --out {RES}/timing_verification_this_machine_cpu.json",
    log=f"{RES}/verify_timing_cpu.log")
if RUN_CPU_REFERENCE:
    run(f"python scripts/evaluate_research_extension.py --device cpu --checkpoint-dir {CPU_REF} --out-dir {CPU_REF}/results --deim-sweep {SWEEP}",
        log=f"{CPU_REF}/evaluate_research_extension.log")

# MLP seeds (the repository's multi-seed evaluator covers the FNO only): in-distribution error per seed
mlp_seeds = {s: FULL["mlp"][s]["test_rel_l2_mean"] for s in SEEDS}
json.dump({"seeds": SEEDS, "in_distribution_per_seed": mlp_seeds,
           "mean_across_seeds": float(np.mean(list(mlp_seeds.values()))),
           "std_across_seeds": float(np.std(list(mlp_seeds.values()))), "device": "cuda"},
          open(f"{RES}/mlp_multiseed_results.json", "w"), indent=2)

## 10. Integrity: source tree unchanged, checkpoint portability

In [ ]:
assert GATE_PASSED, "run the gate (section 7) first"
# Tracked files must match the commit, and no new non-ignored file may appear. Generated outputs
# (data/*.npz, experiments/*.npz, runs/, __pycache__) are git-ignored by design and listed separately.
source_changes = run("git status --porcelain").strip()
if source_changes != DIRTY_AT_START:
    raise RuntimeError(f"the experiment modified the source tree:\n{source_changes}")
generated = [l[3:] for l in run("git status --porcelain --ignored").splitlines() if l.startswith("!! ")]
print("source tree unchanged since checkout; ignored generated paths:", generated)
with open(f"{RUN}/git_status.txt", "w") as f:
    f.write(f"commit {COMMIT}\nsource changes: {source_changes or 'none'}\nignored generated paths:\n" + "\n".join(generated))

# a CUDA-saved checkpoint loads on CPU through map_location and predicts the same field
nets = []
for dev in ("cuda", "cpu"):
    c = torch.load(f"{CK}/fno_checkpoint.pt", map_location=dev, weights_only=True)
    m = FNO1d(c["Nx"], c["n_save"], modes=c["modes"], width=c["width"], n_layers=c["n_layers"]).to(dev).eval()
    m.load_state_dict(c["model_state"]); nets.append(m)
u0 = torch.from_numpy(TEST["u"][:, 0].astype(np.float32))
with torch.no_grad():
    PORT_DIFF = (nets[0](u0.cuda()).cpu() - nets[1](u0)).abs().max().item()
print(f"CUDA checkpoint on CPU: max |CUDA - CPU prediction| = {PORT_DIFF:.2e}")
if PORT_DIFF > 1e-3:
    raise RuntimeError(PORT_DIFF)

## 11. Metrics and provenance

Three results, side by side and not ranked: the committed MPS result on the historical data, this CUDA run, and this machine's CPU on exactly the same data as the CUDA run. Accuracy differences between backends at the size of the seed-to-seed spread are noise. Timings are hardware-specific and do not replace the committed CPU/MPS timings.

In [ ]:
assert GATE_PASSED, "run the gate (section 7) first"
r = verify_results(RES)
ms = json.load(open(f"{RES}/fno_multiseed_results.json"))
tc = json.load(open(f"{RES}/timing_verification_cuda.json"))
tp = json.load(open(f"{RES}/timing_verification_this_machine_cpu.json"))
hist = json.load(open("report/research/results.json"))  # committed Apple MPS reference, read-only
rc = json.load(open(f"{CPU_REF}/results/results.json")) if RUN_CPU_REFERENCE else None
hist_label = "committed MPS" if DATA_PROV["like_for_like_with_historical_mps"] else "MPS (other data!)"

print(f"data: {DATA_SOURCE} | like-for-like with the committed MPS results: {DATA_PROV['like_for_like_with_historical_mps']}")
print(f"\nIn-distribution relative L2, seed 0 (mean over {r['in_distribution']['mlp']['n']} test cases)")
print(f"{'method':9s} {hist_label:>18s} {'CUDA (this run)':>16s} {'CPU (this run)':>15s}")
for m in ("rom", "deim_rom", "mlp", "fno"):
    print(f"{m:9s} {hist['in_distribution'][m]['mean']:18.5f} {r['in_distribution'][m]['mean']:16.5f} "
          + (f"{rc['in_distribution'][m]['mean']:15.5f}" if rc else f"{'-':>15s}"))
print(f"{'fno OOD':9s} {hist['out_of_family_ic']['fno']['mean']:18.5f} {r['out_of_family_ic']['fno']['mean']:16.5f} "
      + (f"{rc['out_of_family_ic']['fno']['mean']:15.5f}" if rc else f"{'-':>15s}"))

print(f"\nSeeds {SEEDS} on CUDA, in-distribution mean +/- std across seeds:")
print(f"  MLP {100*np.mean(list(mlp_seeds.values())):.3f}% +/- {100*np.std(list(mlp_seeds.values())):.3f}%")
print(f"  FNO {100*ms['in_distribution']['mean_across_seeds']:.3f}% +/- {100*ms['in_distribution']['std_across_seeds']:.3f}%")

print("\nTraining time on this machine, seed 0 (s)")
for label, res in [("CUDA", FULL)] + ([("CPU", {"mlp": {0: CPU["mlp"]}, "fno": {0: CPU["fno"]}})] if CPU else []):
    print(f"  {label:5s} MLP {res['mlp'][0]['meta']['total_train_time_s']:7.2f}   FNO {res['fno'][0]['meta']['total_train_time_s']:7.2f}")

print("\nOnline cost on this machine, median of 7 trials x 20 calls (ms, batch 1, incl. host copy)")
for m in ("solver", "rom", "deim_rom", "mlp", "fno"):
    if m in ("mlp", "fno"):
        print(f"  {m:9s} CUDA {tc['timing_ms'][m]['median']:8.3f}   CPU {tp['timing_ms'][m]['median']:8.3f}")
    else:
        print(f"  {m:9s} NumPy CPU {tc['timing_ms'][m]['median']:8.3f}")

assert RUN_ID, "RUN_ID must be set (section 3)"
PROV = {"run_id": RUN_ID, "repo": REPO_URL, "ref": REF, "commit": COMMIT, "dirty_at_start": bool(DIRTY_AT_START),
        "device_info": r["device_info"], "data": DATA_PROV, "seeds": SEEDS, "cpu_reference": RUN_CPU_REFERENCE,
        "checkpoint_portability_max_abs_diff": PORT_DIFF,
        "python": sys.version.split()[0], "numpy": np.__version__, "torch": torch.__version__,
        "note": "CUDA backend experiment; it does not replace the committed CPU/MPS results in report/."}
json.dump(PROV, open(f"{RUN}/provenance.json", "w"), indent=2)
print(json.dumps(PROV, indent=2))

## 12. Package `runs/<RUN_ID>/` (checkpoints, metrics, plots, logs, metadata)

In [ ]:
assert GATE_PASSED, "run the gate (section 7) first"
run(f"nvidia-smi > {RUN}/nvidia_smi.txt")
run(f"pip freeze > {RUN}/pip_freeze.txt")
assert RUN_ID and os.path.isdir(f"runs/{RUN_ID}"), f"no run directory for RUN_ID={RUN_ID!r}"
archive = shutil.make_archive(f"/content/{RUN_ID}", "zip", "runs", RUN_ID)  # this run only
print(archive, f"{os.path.getsize(archive) / 2**20:.1f} MB")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("not on Colab: download", archive)